# Basis-resolution rule: the fixed WIDTH and smallest NBASIS that represent every KBM field of the database

Fusion_PhD-qp5.43 (KBM paper, section "Resolution with incorrect width"). The chain is the paper's: Gaussian $\to$ example KBM $\to$ database $\to$ rule.

**Method.** Each GS2 KBM's complex $\phi(\theta)$ and $A_\parallel(\theta)$ is projected onto the Hermite basis that TGLF and GFTM use, and the
relative $L^2$ truncation error $\varepsilon_f(W,N)=\lVert f-P_Nf\rVert/\lVert f\rVert$ is measured for every WIDTH $W$ and every number of basis functions $N$.
No extent-to-width conversion is needed for that.

**The basis, from the source** (`tglf_hermite.f90`, `tglf_geometry.f90`, `tglf_LS.f90`; the GFTM files are the same lines): functions
$\psi_j(x)\propto H_j(x)e^{-x^2/2}$ with $x=\theta_b/\mathrm{WIDTH}$. $\theta_b=2\pi y/L_y$ is **not** the poloidal angle: $y=\int (B/B_p)\,dl_p$ is
the arc length along the field line. Both codes use the same definition, so WIDTH means the same in TGLF and GFTM (no factor $\sqrt2$).
The map $\theta\to\theta_b$ is computed here from each sample's pyrokinetics `local_geometry`.

**Population**: the 738 GS2-KBMs of the four `kbm_free` Latin hypercubes (classifier cell copied from `fixed_width_nbasis_convergence.ipynb`).
Growth rates and the classification come from `kbm_free/pyroscan.nc` (time average); the complex fields from `kbm_free_complex/pyroscan.nc`
(the same runs re-saved by pyrokinetics with `time_mode="last"`, `results/basis_resolution_rule/runs/resave_gs2_complex.py`).

**Defaults that are scientific choices (flagged for the user):** target error $\varepsilon=1\%$ (5 % and 0.1 % also shown), quantile 90 % of samples,
fields $\phi$ and $A_\parallel$ (not $B_\parallel$), error norm relative $L^2$ in $\theta_b$, basis centred at $\theta=0$.

In [ ]:
import os
for threads in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[threads] = "1"
import json
import re
from math import comb
from multiprocessing import get_context
from pathlib import Path
import warnings
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import NullLocator, StrMethodFormatter
from scipy.interpolate import CubicSpline
import pyrokinetics
from pyrokinetics import PyroHypercube
from pyrokinetics.diagnostics.extent import Extent
from general_analysis import mode_classification as mc

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir())
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
# the style of the paper's Basis_Function_requirements.png (10.6 x 8.0 in at 200 dpi), copied from hermite_projections_of_gaussian.ipynb
talk_style = {"figure.figsize": (10.6, 8.0), "savefig.dpi": 200, "savefig.bbox": "standard", "figure.constrained_layout.use": True,
              "font.size": 18, "axes.labelsize": 22, "axes.titlesize": 22, "figure.titlesize": 22,
              "xtick.labelsize": 18, "ytick.labelsize": 18, "legend.fontsize": 18, "lines.linewidth": 2.5, "lines.marker": "",
              "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white"}
load_dotenv(ROOT / "local.env", override=True)
warnings.filterwarnings("ignore")
print("pyrokinetics", pyrokinetics.__version__, pyrokinetics.__file__)

data_root = Path(os.path.expandvars(os.environ["GK_DATA_ROOT"]))     # the WORK tree
plots_dir = ROOT / "Plots" / "TGLF_GFTM_KBM_PAPER" / "basis_resolution_rule"
csv_dir = Path("csv")
plots_dir.mkdir(parents=True, exist_ok=True); csv_dir.mkdir(exist_ok=True)
cases = ["SPR-045", "M1", "R4", "R3"]
example_case, example_sample = "SPR-045", "sample_0818"              # the paper's example KBM
fields = ["phi", "apar"]
field_label = {"phi": r"$\phi$", "apar": r"$A_\parallel$"}
tolerances = [0.05, 0.01, 0.001]                                     # the three lines of Basis_Function_requirements.png
tolerance_label = {0.05: "5%", 0.01: "1%", 0.001: "0.1%"}
target_tolerance, target_quantile = 0.01, 0.90                       # the rule's defaults (user, 2026-11-13); both are the user's to change
validation_tolerance = 0.10                                          # projection error called "resolved" in the comparison with the codes (1% is reached by too few samples to plot)
code_widths = [0.2, 0.2297, 0.2639, 0.3031, 0.3482, 0.4, 0.4595, 0.5279, 0.6063, 0.6965, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6,
               1.9218, 2.3083, 2.7726, 3.3302, 4.0]                  # registry.WIDTH_ORACLE_GRID, the widths the codes were run at
projection_widths = np.unique(np.concatenate([code_widths, np.round(np.geomspace(0.05, 8, 90), 4)]))
n_max = 256                                                          # largest N projected (GFTM's own cap is 64, TGLF's 32)
width_tag = lambda width: f"w{width:.4f}".replace(".", "p")
tglf_nbasis = [2, 4, 6, 8, 12, 16, 24]                               # FILTER -1.5 ladder, NXGRID 32
gftm_cells = {2: 32, 4: 32, 8: 32, 12: 32, 16: 32, 24: 32, 32: 40, 48: 65, 56: 64}   # NBASIS: NXGRID (NXGRID >= NBASIS + 8), FILTER 0.5
n_cpu = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))
magnitude = lambda quantity: np.asarray(getattr(quantity, "m", quantity))

## The Hermite basis and the Gaussian model
Orthonormal Hermite functions by the codes' own recurrence (`tglf_hermite.f90` lines 127-132). Coefficients by the trapezoid rule on a uniform
$x$ grid, which is spectrally accurate for these integrands. `gaussian_error` is the closed form of `hermite_projections_of_gaussian.ipynb`.
The check at the end runs a Gaussian through the same projection used for the GS2 fields and compares with the closed form.

In [ ]:
x_grid = np.arange(-(np.sqrt(2 * n_max + 1) + 6), np.sqrt(2 * n_max + 1) + 6, 0.01)   # psi_255 has turning points at |x| = 22.6
dx = x_grid[1] - x_grid[0]


def hermite_functions(order_count, x):
    functions = np.zeros((order_count, len(x)))
    functions[0] = np.pi ** -0.25 * np.exp(-x ** 2 / 2)
    functions[1] = np.sqrt(2) * x * functions[0]
    for n in range(2, order_count):
        functions[n] = np.sqrt(2 / n) * x * functions[n - 1] - np.sqrt((n - 1) / n) * functions[n - 2]
    return functions


hermite_x = hermite_functions(n_max, x_grid)
# Gauss-Hermite nodes as the codes use them (nx = 2 NXGRID - 1 roots of H_nx); weight for a plain integral is 1 / (nx psi_{nx-1}(x_i)^2)
quadrature = {}
for nxgrid in sorted(set(gftm_cells.values())):
    nodes = np.polynomial.hermite.hermgauss(2 * nxgrid - 1)[0]
    functions_at_nodes = hermite_functions(2 * nxgrid - 1, nodes)
    quadrature[nxgrid] = (nodes, 1 / ((2 * nxgrid - 1) * functions_at_nodes[-1] ** 2), functions_at_nodes)
quadrature_cells = [("TGLF", nbasis, 32) for nbasis in tglf_nbasis if nbasis >= 8] + [("GFTM", nbasis, nxgrid) for nbasis, nxgrid in gftm_cells.items() if nbasis >= 32]


def projection_errors(basis_angle, values):
    # relative L2 error of the projection of f(theta_b) onto the first N Hermite functions of every projection width: array (width, N)
    # second output: the same with the coefficients taken by the code's Gauss-Hermite quadrature, for each quadrature cell
    spline = CubicSpline(basis_angle, values)
    fine_angle = np.arange(basis_angle[0], basis_angle[-1], 0.005)
    norm = np.sum(np.abs(spline(fine_angle)) ** 2) * 0.005
    inside = lambda angle: np.where((angle >= basis_angle[0]) & (angle <= basis_angle[-1]), spline(np.clip(angle, basis_angle[0], basis_angle[-1])), 0)
    error = np.ones((len(projection_widths), n_max), np.float32)
    error_quadrature = np.ones((len(projection_widths), len(quadrature_cells)), np.float32)
    for i, width in enumerate(projection_widths):
        coefficients = np.sqrt(width) * dx * (hermite_x @ inside(width * x_grid))
        missed = np.clip(1 - np.cumsum(np.abs(coefficients) ** 2) / norm, 0, None)
        error[i] = np.sqrt(missed)
        for j, (code, nbasis, nxgrid) in enumerate(quadrature_cells):
            nodes, weights, functions_at_nodes = quadrature[nxgrid]
            coefficients_quadrature = np.sqrt(width) * (functions_at_nodes[:nbasis] * weights) @ inside(width * nodes)
            error_quadrature[i, j] = np.sqrt(missed[nbasis - 1] + np.sum(np.abs(coefficients_quadrature - coefficients[:nbasis]) ** 2) / norm)
    return error, error_quadrature


def needed_nbasis(error, tolerance):
    # smallest N with error <= tolerance along the last axis; inf where n_max functions are not enough
    reached = error <= tolerance
    return np.where(reached.any(-1), reached.argmax(-1) + 1, np.inf)


def gaussian_error(s, n_basis):
    r_squared = ((s ** 2 - 1) / (s ** 2 + 1)) ** 2
    kept = sum(comb(2 * k, k) * (r_squared / 4) ** k for k in range((n_basis + 1) // 2))
    return np.sqrt(max(1 - np.sqrt(1 - r_squared) * kept, 0))


model_angle = np.linspace(-40, 40, 8001)
model_error = {"phi": projection_errors(model_angle, np.exp(-model_angle ** 2 / 2) + 0j)[0],                 # even Gaussian, sigma = 1
               "apar": projection_errors(model_angle, model_angle * np.exp(-model_angle ** 2 / 2) + 0j)[0]}  # odd x Gaussian, sigma = 1
for width, n_basis in [(0.5, 8), (0.5, 16), (2.0, 8), (3.0, 40)]:
    measured = model_error["phi"][np.argmin(np.abs(projection_widths - width)), n_basis - 1]
    exact = gaussian_error(1 / projection_widths[np.argmin(np.abs(projection_widths - width))], n_basis)
    assert abs(measured - exact) < 2e-5, (width, n_basis, measured, exact)
print("projection of a Gaussian reproduces the closed form")

## GS2 reference, population, and the projection of every KBM
The classifier cell is the one of `fixed_width_nbasis_convergence.ipynb`. Then, per KBM sample: the map $\theta\to\theta_b$ from the sample's
Miller geometry (`get_flux_surface`, `get_b_poloidal`, `get_dLdtheta`, `Fpsi`), the complex field on $\theta_b$, and its projection errors.
The extents are pyro's 95 % `Extent` of $|f|^2$, in $\theta$ and in $\theta_b$.

In [ ]:
theta_period = np.linspace(0, 2 * np.pi, 2001)


def tied_species(sample):
    # GyroRun's links for the kbm_free cubes (not stored in pyroscan.json): ions share ion1's a/L_T, every a/L_n is the electron's
    species = gs2_scan.sample_pyro(sample).local_species
    ions = [name for name in species.names if name != "electron"]
    for ion in ions[1:]:
        species[ion].inverse_lt = species[ions[0]].inverse_lt
    for ion in ions:
        species[ion].inverse_ln = species["electron"].inverse_ln
    species.enforce_quasineutrality("electron")
    return {name: (float(species[name].temp.m), float(species[name].dens.m), float(species[name].inverse_lt.m)) for name in species.names}


def project_sample(sample_index):
    geometry = gs2_scan.sample_pyro(sample_index).local_geometry
    major_radius = magnitude(geometry.get_flux_surface(theta_period)[0])
    b_over_b_poloidal = np.sqrt(1 + (magnitude(geometry.Fpsi) / major_radius / magnitude(geometry.get_b_poloidal(theta_period))) ** 2)
    arc_integrand = b_over_b_poloidal * magnitude(geometry.get_dLdtheta(theta_period))        # dy/dtheta, y = arc length along the field
    arc_length = np.concatenate([[0], np.cumsum(0.5 * (arc_integrand[1:] + arc_integrand[:-1]) * np.diff(theta_period))])
    basis_angle_period = 2 * np.pi * arc_length / arc_length[-1]
    out = dict(stretch=basis_angle_period[1] / theta_period[1])                                # d theta_b / d theta at theta = 0
    for field in fields:
        values = complex_fields[field][sample_index]
        theta = gs2_theta[np.isfinite(values)]
        turns = np.floor(theta / (2 * np.pi))
        basis_angle = 2 * np.pi * turns + np.interp(theta - 2 * np.pi * turns, theta_period, basis_angle_period)
        out[field], out[f"{field}_quadrature"] = projection_errors(basis_angle, values[np.isfinite(values)])
        power = np.abs(values[np.isfinite(values)]) ** 2
        weight = np.gradient(basis_angle)
        out[f"{field}_tail_fraction"] = float((power * weight)[np.abs(basis_angle) > np.pi].sum() / (power * weight).sum())   # |f|^2 beyond one poloidal half-turn
        out[f"{field}_grid_roughness"] = float(np.sqrt((np.abs(np.diff(values[np.isfinite(values)], 2)) ** 2).sum() / power.sum()))   # second difference on the GS2 grid, relative
        out[f"{field}_extent_theta"] = float(Extent._compute(xr.DataArray(power[None], dims=("sample", "theta"), coords=dict(theta=theta)), 0.95)[0][0])
        out[f"{field}_extent_basis"] = float(Extent._compute(xr.DataArray(power[None], dims=("sample", "theta"), coords=dict(theta=basis_angle)), 0.95)[0][0])
    return out


gs2, projected = {}, {}
for case in cases:
    leaf = data_root / "GS2/Runs/LATIN_HYPERCUBE" / case / "kbm_free"
    gs2_scan = PyroHypercube(pyroscan_json=leaf / "pyroscan.json", load_base_pyro=True)
    gs2_scan.load_gk_output(netcdf_file=leaf / "pyroscan.nc")
    gs2_output = gs2_scan.gk_output.data
    names = np.array([str(name) for name in gs2_output.sample_name.values])
    with get_context("fork").Pool(n_cpu) as pool:
        species = pool.map(tied_species, range(len(names)), chunksize=8)
    heat = gs2_output.heat.sum("field").pint.dequantify()
    particle = gs2_output.particle.sum("field").pint.dequantify()

    def chi(group):   # (Q_s - 1.5 T_s Gamma_s) / (n_s T_s a/L_Ts) summed over the group, as mode_classification.indicators
        numerator = denominator = 0.0
        for name in group:
            temp, dens, inverse_lt = np.array([sp[name] for sp in species]).T
            numerator = numerator + heat.sel(species=name).values - 1.5 * temp * particle.sel(species=name).values
            denominator = denominator + dens * temp * inverse_lt
        return numerator / denominator
    ions = [str(s) for s in gs2_output.species.values if s != "electron"]
    with np.errstate(divide="ignore", invalid="ignore"):
        labels = np.asarray(mc.classify(dict(T=gs2_output.tearing_parameter.values.ravel(),
                                             omega=mc.ION_DIRECTION * gs2_output.mode_frequency.pint.dequantify().values.ravel(),
                                             chi_ratio=chi(["electron"]) / chi(ions))))
    gamma = gs2_output.growth_rate.pint.dequantify().values.ravel()
    tolerance = gs2_output.growth_rate_tolerance.pint.dequantify().values.ravel()
    kbm = np.isfinite(gamma) & (gamma > 0) & (tolerance < 0.05) & (labels == "KBM")
    gs2[case] = dict(names=names[kbm], gamma=gamma[kbm], omega=mc.ION_DIRECTION * gs2_output.mode_frequency.pint.dequantify().values.ravel()[kbm],
                     units=re.sub(r"_pyroscan_base\d+", "", str(gs2_output.growth_rate.data.units)))

    complex_leaf = data_root / "GS2/Runs/LATIN_HYPERCUBE" / case / "kbm_free_complex"     # same runs, complex final-time fields
    gs2_scan.load_gk_output(netcdf_file=complex_leaf / "pyroscan.nc")
    gs2_complex = gs2_scan.gk_output.data
    assert [str(name) for name in gs2_complex.sample_name.values] == list(names), "complex cube is not the same scan"
    gs2_theta = gs2_complex.theta.values
    complex_fields = {field: gs2_complex[field].pint.dequantify().squeeze("kx").transpose("sample", "theta").values for field in fields}
    with get_context("fork").Pool(n_cpu) as pool:
        projected[case] = pool.map(project_sample, np.flatnonzero(kbm), chunksize=4)
    print(case, "GS2-KBMs", kbm.sum(), "| median 95% extent in theta: phi", np.median([p["phi_extent_theta"] for p in projected[case]]),
          "apar", np.median([p["apar_extent_theta"] for p in projected[case]]), "| median d theta_b/d theta at 0:", np.median([p["stretch"] for p in projected[case]]), flush=True)
    del gs2_output, gs2_complex, complex_fields

samples = pd.concat([pd.DataFrame(dict(case=case, sample=gs2[case]["names"], gamma_gs2=gs2[case]["gamma"], stretch=[p["stretch"] for p in projected[case]],
                                       **{f"{field}_{quantity}": [p[f"{field}_{quantity}"] for p in projected[case]] for field in fields for quantity in ("extent_theta", "extent_basis", "tail_fraction", "grid_roughness")}))
                     for case in cases], ignore_index=True)
assert len(samples) == 738, len(samples)
error = {field: np.stack([p[field] for case in cases for p in projected[case]]) for field in fields}                       # (sample, width, N)
error_quadrature = {field: np.stack([p[f"{field}_quadrature"] for case in cases for p in projected[case]]) for field in fields}
error["both"] = np.maximum(error["phi"], error["apar"])
example = int(np.flatnonzero((samples.case == example_case) & (samples["sample"] == example_sample))[0])
del projected

# Gaussian-equivalent width of each field: the WIDTH at which ONE basis function captures most of it (psi_0 for phi, psi_1 for A_par)
single_function_error = {"phi": error["phi"][:, :, 0], "apar": np.sqrt(np.clip(1 - (error["apar"][:, :, 0] ** 2 - error["apar"][:, :, 1] ** 2), 0, None))}
for field in fields:
    samples[f"{field}_sigma"] = projection_widths[single_function_error[field].argmin(1)]
    samples[f"{field}_single_function_error"] = single_function_error[field].min(1)
model_sigma = {field: projection_widths[(model_error[field][:, 0] if field == "phi" else -(model_error[field][:, 0] ** 2 - model_error[field][:, 1] ** 2)).argmin()] for field in fields}
print("model sigma check (should be 1):", model_sigma)
print(samples.groupby("case")[["stretch", "phi_extent_theta", "phi_extent_basis", "apar_extent_theta", "apar_extent_basis", "phi_sigma", "apar_sigma",
                               "phi_single_function_error", "apar_single_function_error", "phi_tail_fraction", "apar_tail_fraction", "phi_grid_roughness", "apar_grid_roughness"]].median().round(3).to_string())
print("example KBM:", samples.loc[example].to_dict())

## Step 1, Gaussian $\to$ example KBM (paper subsection "Error in capturing an Example Kinetic Ballooning Mode")
The paper's `Basis_Function_requirements` figure, same axes and lines (the bound $E_M=\lvert(S^2-1)/(S^2+1)\rvert^M$ for 5 %, 1 %, 0.1 %), with the measured
requirement of the example KBM `sample_0818` added as dashed lines with points. $S=\sigma/W$, where $\sigma$ is the mode's Gaussian-equivalent width:
the WIDTH at which a single basis function captures most of the field. For $\phi$ (even) $M$ counts even functions, $M=\lceil N/2\rceil$. For
$A_\parallel$ (odd) the sibling figure counts odd functions, $M=\lfloor N/2\rfloor$, and the solid lines are the measured requirement of the odd model
$x\,e^{-x^2/2\sigma^2}$ (there is no bound in the paper for it).

In [ ]:
s_line = np.geomspace(0.1, 10, 400)
count = {"phi": lambda nbasis: np.ceil(nbasis / 2), "apar": lambda nbasis: np.floor(nbasis / 2)}
count_label = {"phi": "M Even Hermite Functions", "apar": "M Odd Hermite Functions"}


def model_lines(ax, field):
    for tolerance, colour in zip(tolerances, ["C0", "C1", "C2"]):
        if field == "phi":
            ax.plot(s_line, np.log(tolerance) / np.log(np.abs((s_line ** 2 - 1) / (s_line ** 2 + 1))), color=colour, label=tolerance_label[tolerance])
        else:
            needed = count[field](needed_nbasis(model_error[field], tolerance))
            ax.plot(1 / projection_widths[np.isfinite(needed)], needed[np.isfinite(needed)], color=colour, label=tolerance_label[tolerance])
    ax.set_xscale("log"); ax.set(xlabel="$S$", ylabel=count_label[field], xlim=(0.08, 12.5))
    ax.grid(True, which="both", alpha=0.5)


with plt.rc_context(talk_style):
    for field in fields:
        fig, ax = plt.subplots()
        model_lines(ax, field)
        for tolerance, colour in zip(tolerances, ["C0", "C1", "C2"]):
            needed = count[field](needed_nbasis(error[field][example], tolerance))
            ax.plot(samples[f"{field}_sigma"][example] / projection_widths[np.isfinite(needed)], needed[np.isfinite(needed)], color=colour, ls="--", marker="o", ms=5)
        handles = ax.get_legend_handles_labels()[0] + [Line2D([], [], color="k"), Line2D([], [], color="k", ls="--", marker="o", ms=5)]
        ax.legend(handles, [tolerance_label[t] for t in tolerances] + ["Gaussian" if field == "phi" else r"$x\,\times$ Gaussian", f"KBM {field_label[field]}"], loc="upper center")
        for suffix in ("pdf", "png"):
            fig.savefig(plots_dir / f"Basis_Function_requirements_example_kbm_{field}.{suffix}")
        plt.close(fig)
        for tolerance in tolerances:
            needed = needed_nbasis(error[field][example], tolerance)
            print(f"example KBM {field} eps {tolerance}: sigma {samples[f'{field}_sigma'][example]}, min N {np.nanmin(needed)} at WIDTH {projection_widths[np.argmin(needed)]}",
                  "| N at code widths 0.3482/0.6063/1.0/2.3083:", [needed[np.argmin(np.abs(projection_widths - w))] for w in (0.3482, 0.6063, 1.0, 2.3083)])

## Step 2, example KBM $\to$ database (paper subsection for the database)
The same figure with the 738 GS2-KBMs: per $S$, the median of the measured requirement over the samples (dashed) and the 10-90 % band. Each sample
uses its own $\sigma$, so the horizontal axis needs no extent conversion.

In [ ]:
log_s_bins = np.linspace(np.log10(0.1), np.log10(10), 41)
with plt.rc_context(talk_style):
    for field in fields:
        fig, ax = plt.subplots()
        model_lines(ax, field)
        log_s = np.log10(samples[f"{field}_sigma"].values[:, None] / projection_widths[None, :])
        for tolerance, colour in zip(tolerances, ["C0", "C1", "C2"]):
            needed = count[field](needed_nbasis(error[field], tolerance))
            centre, low, median, high = [], [], [], []
            for left, right in zip(log_s_bins[:-1], log_s_bins[1:]):
                in_bin = (log_s >= left) & (log_s < right)
                if in_bin.sum() >= 100:
                    centre.append(10 ** ((left + right) / 2))
                    low.append(np.quantile(needed[in_bin], 0.1, method="higher")); median.append(np.quantile(needed[in_bin], 0.5, method="higher")); high.append(np.quantile(needed[in_bin], 0.9, method="higher"))
            centre, low, median, high = map(np.array, (centre, low, median, high))
            ax.plot(centre[np.isfinite(median)], median[np.isfinite(median)], color=colour, ls="--")
            ax.fill_between(centre, low, np.where(np.isfinite(high), high, n_max / 2), where=np.isfinite(low), color=colour, alpha=0.2, lw=0)
        handles = ax.get_legend_handles_labels()[0] + [Line2D([], [], color="k"), Line2D([], [], color="k", ls="--")]
        ax.legend(handles, [tolerance_label[t] for t in tolerances] + ["Gaussian" if field == "phi" else r"$x\,\times$ Gaussian", f"KBM database {field_label[field]}"], loc="upper center")
        for suffix in ("pdf", "png"):
            fig.savefig(plots_dir / f"Basis_Function_requirements_database_{field}.{suffix}")
        plt.close(fig)

# How many functions the database needs at the best width for each sample, against a Gaussian (which needs 1 or 2)
for field in fields:
    best_5pct = pd.Series(np.where(np.isinf(needed_nbasis(error[field], 0.05).min(1)), 2 * n_max, needed_nbasis(error[field], 0.05).min(1)))
    print(f"{field}: Spearman rank correlation of the N needed for 5% (own best WIDTH) with the tail fraction {best_5pct.corr(samples[f'{field}_tail_fraction'], method='spearman'):.2f},",
          f"with the grid roughness {best_5pct.corr(samples[f'{field}_grid_roughness'], method='spearman'):.2f}, with the single-function error {best_5pct.corr(samples[f'{field}_single_function_error'], method='spearman'):.2f}")
    for tolerance in tolerances:
        needed = needed_nbasis(error[field], tolerance)
        best = needed.min(1)
        print(f"{field} eps {tolerance}: N at each sample's own best WIDTH: median {np.median(best)}, 10-90% {np.quantile(best, 0.1)}-{np.quantile(best, 0.9)},",
              f"not reached with {n_max}: {np.isinf(best).sum()}; best WIDTH / sigma median {np.median(projection_widths[needed.argmin(1)] / samples[f'{field}_sigma']):.2f}")

## How well does the extent predict the Gaussian-equivalent width?
For a Gaussian amplitude $e^{-\theta_b^2/2\sigma^2}$ the 95 % extent of $|f|^2$ is $2\times1.96\,\sigma/\sqrt2=2.77\sigma$. For the odd model
$x\,e^{-x^2/2\sigma^2}$ it is $4.32\sigma$ (measured below on the model). The figure compares each sample's $\sigma$ with its extent, both in $\theta_b$.

In [ ]:
model_power = (model_angle * np.exp(-model_angle ** 2 / 2)) ** 2
extent_per_sigma = {"phi": 2 * 1.959964 / np.sqrt(2),
                    "apar": float(Extent._compute(xr.DataArray(model_power[None], dims=("sample", "theta"), coords=dict(theta=model_angle)), 0.95)[0][0])}
print("extent / sigma of the models:", extent_per_sigma)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), constrained_layout=True)
for ax, field in zip(axes, fields):
    for case in cases:
        rows = samples[samples.case == case]
        ax.plot(rows[f"{field}_extent_basis"], rows[f"{field}_sigma"], "o", ms=3, alpha=0.6, label=case)
    extent_line = np.array([samples[f"{field}_extent_basis"].min(), samples[f"{field}_extent_basis"].max()])
    ax.plot(extent_line, extent_line / extent_per_sigma[field], "k--", label="Gaussian model")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_title(field_label[field], fontsize=16); ax.set_xlabel(r"95% extent in $\theta_b$", fontsize=16); ax.set_ylabel(r"$\sigma$ (best single-function WIDTH)", fontsize=16)
    ax.tick_params(axis="both", labelsize=14)
    ratio = samples[f"{field}_sigma"] * extent_per_sigma[field] / samples[f"{field}_extent_basis"]
    print(f"{field}: sigma / (extent / {extent_per_sigma[field]:.2f}) median {ratio.median():.3f}, 10-90% {ratio.quantile(0.1):.3f}-{ratio.quantile(0.9):.3f};",
          f"extent in theta_b / extent in theta median {(samples[f'{field}_extent_basis'] / samples[f'{field}_extent_theta']).median():.3f}")
axes[0].legend(fontsize=12)
for suffix in ("pdf", "png"):
    fig.savefig(plots_dir / f"sigma_vs_extent.{suffix}")
plt.close(fig)

# Does the Gaussian model, fed each sample's sigma, predict the measured requirement? Ratio measured N / model N at the code widths.
for field in fields:
    for tolerance in (0.05, 0.01):
        model_needed = needed_nbasis(model_error[field], tolerance)
        measured = needed_nbasis(error[field], tolerance)
        log_s_model = np.log(model_sigma[field] / projection_widths)
        predicted = np.array([[np.interp(np.log(sigma / width), log_s_model[::-1], np.where(np.isfinite(model_needed), model_needed, n_max)[::-1]) for width in code_widths] for sigma in samples[f"{field}_sigma"]])
        at_code_widths = measured[:, [int(np.argmin(np.abs(projection_widths - width))) for width in code_widths]]
        ratio = at_code_widths / np.maximum(predicted, 1)
        print(f"{field} eps {tolerance}: measured N / Gaussian-model N, median per code width:", dict(zip(code_widths, np.round(np.median(ratio, 0), 2))))

## Step 3, database $\to$ rule
For each WIDTH, the smallest $N$ such that a fraction $q$ of the 738 samples has $\varepsilon\le$ tolerance for **both** fields. The rule's WIDTH is the
minimum of that curve. Panels: the default 1 %, then 5 % and 10 %. Solid: both fields, 50 % and 90 % of samples. Dashed: $\phi$ alone and $A_\parallel$ alone at 90 %.

In [ ]:
def quantile_needed(field, tolerance, quantile, rows=slice(None)):
    return np.quantile(needed_nbasis(error[field][rows], tolerance), quantile, axis=0, method="higher")


fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharex=True, sharey=True, constrained_layout=True)
for ax, tolerance in zip(axes, (0.01, 0.05, 0.1)):
    for quantile, colour in ((0.5, "C0"), (0.9, "C3")):
        needed = quantile_needed("both", tolerance, quantile)
        ax.plot(projection_widths[np.isfinite(needed)], needed[np.isfinite(needed)], color=colour, label=f"both fields, {quantile:.0%} of samples")
    for field, style in (("phi", "--"), ("apar", ":")):
        needed = quantile_needed(field, tolerance, 0.9)
        ax.plot(projection_widths[np.isfinite(needed)], needed[np.isfinite(needed)], color="C3", ls=style, label=f"{field_label[field]}, 90%")
    ax.set_xscale("log"); ax.set_yscale("log", base=2)
    ax.set_title(f"error {tolerance:.0%}", fontsize=16); ax.set_xlabel("WIDTH", fontsize=16); ax.tick_params(axis="both", labelsize=14)
    ax.set_xticks([0.1, 0.2, 0.5, 1, 2, 4]); ax.xaxis.set_major_formatter(StrMethodFormatter("{x:g}")); ax.xaxis.set_minor_locator(NullLocator())
    ax.set_yticks([2, 4, 8, 16, 32, 64, 128, 256]); ax.yaxis.set_major_formatter(StrMethodFormatter("{x:g}")); ax.yaxis.set_minor_locator(NullLocator())
axes[0].set_xlim(0.08, 5); axes[0].set_ylabel("NBASIS needed", fontsize=16); axes[0].legend(fontsize=12)
for suffix in ("pdf", "png"):
    fig.savefig(plots_dir / f"nbasis_needed_vs_width.{suffix}")
plt.close(fig)

# The rule and the alternatives. Widths from extents use the Gaussian conversion sigma = extent / 2.77 (phi) or / 4.32 (A_par), extents in theta_b.
sigma_from_extent = {field: samples[f"{field}_extent_basis"] / extent_per_sigma[field] for field in fields}
low = min(sigma_from_extent[field].quantile(0.05) for field in fields)
high = max(sigma_from_extent[field].quantile(0.95) for field in fields)
low_fit = min(samples[f"{field}_sigma"].quantile(0.05) for field in fields)
high_fit = max(samples[f"{field}_sigma"].quantile(0.95) for field in fields)
alternatives = {"user: arithmetic middle of the 90% extent range": (low + high) / 2, "user: geometric middle of the 90% extent range": np.sqrt(low * high),
                "geometric middle of the 90% sigma range": np.sqrt(low_fit * high_fit),
                "geometric mean of the median phi and A_par sigma (t3c)": np.sqrt(samples.phi_sigma.median() * samples.apar_sigma.median()),
                "GFTM recommendation (0.6063)": 0.6063, "GFTM option (0.3482)": 0.3482, "TGLF named setting (1.0)": 1.0}
print(f"90% extent range as sigma: {low:.3f} to {high:.3f} (ratio {high / low:.2f}); 90% sigma range {low_fit:.3f} to {high_fit:.3f} (ratio {high_fit / low_fit:.2f})")
rule_rows = []
for tolerance in (0.1, 0.05, 0.02, 0.01, 0.001):
    for quantile in (0.5, 0.9, 0.95, 1.0):
        needed = quantile_needed("both", tolerance, quantile)
        best = int(np.argmin(needed))
        row = dict(tolerance=tolerance, quantile=quantile, rule_width=projection_widths[best], rule_nbasis=needed[best],
                   width_range_within_25pct_nbasis=f"{projection_widths[needed <= 1.25 * needed[best]].min():g}-{projection_widths[needed <= 1.25 * needed[best]].max():g}")
        for name, width in alternatives.items():
            row[f"N at {name} [{width:.3f}]"] = needed[int(np.argmin(np.abs(projection_widths - width)))]
        rule_rows.append(row)
rule = pd.DataFrame(rule_rows)
rule.to_csv(csv_dir / "rule_summary.csv", index=False)
print(rule.T.to_string())
needed_table = pd.DataFrame({"width": projection_widths, **{f"N_{field}_eps{tolerance}_q{quantile}": quantile_needed(field, tolerance, quantile)
                                                            for field in ("both", "phi", "apar") for tolerance in (0.1, 0.05, 0.01) for quantile in (0.5, 0.9)}})
needed_table.to_csv(csv_dir / "nbasis_needed_vs_width.csv", index=False)
for tolerance in (0.1, 0.05):
    print(f"rule per database at {tolerance:.0%}, 90% (WIDTH, N):", {case: (projection_widths[np.argmin(quantile_needed("both", tolerance, target_quantile, (samples.case == case).values))],
                                                                    quantile_needed("both", tolerance, target_quantile, (samples.case == case).values).min()) for case in cases})

# The rule turned round for an NBASIS the codes can run: the WIDTH that maximises the fraction of samples resolved, and that fraction
feasible_rows = []
for nbasis in (8, 16, 24, 32, 48, 56, 64):
    for tolerance in (0.1, 0.05, 0.03, 0.01):
        for field in ("both", "phi", "apar"):
            fraction = (error[field][:, :, nbasis - 1] <= tolerance).mean(0)
            feasible_rows.append(dict(nbasis=nbasis, tolerance=tolerance, field=field, best_width=projection_widths[fraction.argmax()], fraction_at_best=fraction.max(),
                                      width_range_within_0p02=f"{projection_widths[fraction >= fraction.max() - 0.02].min():g}-{projection_widths[fraction >= fraction.max() - 0.02].max():g}",
                                      **{f"fraction_at_{width}": fraction[int(np.argmin(np.abs(projection_widths - width)))] for width in (0.3482, 0.4595, 0.6063, 1.0)}))
feasible = pd.DataFrame(feasible_rows)
feasible.to_csv(csv_dir / "feasible_rule.csv", index=False)
pd.set_option("display.width", 250)
print(feasible.round(3).to_string())

## The projection error against WIDTH per NBASIS (sibling of `partA_error_vs_width_{TGLF,GFTM}`)
The 90th percentile over the samples of the projection error, one line per NBASIS, in the format of the Part A figures of
`fixed_width_nbasis_convergence.ipynb`. This is what the basis can hold at each of the codes' (WIDTH, NBASIS) cells, before any physics.
The second figure is the ratio of the error with the codes' own Gauss-Hermite quadrature to the exact projection error.

In [ ]:
ladder = [2, 4, 8, 12, 16, 24, 32, 48, 56]
colours = plt.cm.viridis(np.linspace(0, 0.95, len(ladder)))
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharex=True, sharey=True, constrained_layout=True)
for ax, field in zip(axes, ("phi", "apar", "both")):
    for nbasis, colour in zip(ladder, colours):
        ax.plot(projection_widths, np.quantile(error[field][:, :, nbasis - 1], 0.9, axis=0), color=colour, label=f"NBASIS {nbasis}")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_title({"phi": r"$\phi$", "apar": r"$A_\parallel$", "both": "larger of the two"}[field], fontsize=16); ax.set_xlabel("WIDTH", fontsize=16); ax.tick_params(axis="both", labelsize=14)
    ax.xaxis.set_major_formatter(StrMethodFormatter("{x:g}")); ax.xaxis.set_minor_locator(NullLocator())
axes[0].set_xticks([0.2, 0.5, 1, 2, 4]); axes[0].set_xlim(0.19, 4.2); axes[0].set_ylim(0.04, 1.1)
axes[0].set_ylabel("projection error, 90th percentile", fontsize=16); axes[0].legend(fontsize=12, ncol=2)
for suffix in ("pdf", "png"):
    fig.savefig(plots_dir / f"projection_error_vs_width.{suffix}")
plt.close(fig)

for j, (code, nbasis, nxgrid) in enumerate(quadrature_cells):
    exact = np.maximum(error["both"][:, :, nbasis - 1], 1e-6)
    with_quadrature = np.maximum(error_quadrature["phi"][:, :, j], error_quadrature["apar"][:, :, j])
    at = [int(np.argmin(np.abs(projection_widths - width))) for width in (0.3482, 0.6063, 1.0, 2.3083)]
    print(f"{code} NBASIS {nbasis} NXGRID {nxgrid}: 90th-percentile error exact -> code quadrature at WIDTH 0.3482/0.6063/1.0/2.3083:",
          [f"{np.quantile(exact[:, i], 0.9):.4f} -> {np.quantile(with_quadrature[:, i], 0.9):.4f}" for i in at])

## Step 4, validation against the codes (the qp5.37/40 fixed WIDTH $\times$ NBASIS leaves)
The leaves and the loader are those of `fixed_width_nbasis_convergence.ipynb` (TGLF SAT2 FILTER $-1.5$; GFTM stock, FILTER 0.5, NXGRID $\ge$ NBASIS + 8).
Per sample and cell: the dominant mode's $\gamma$, $\omega$ and the 95 % extents, and the projection error of the same sample at the same cell.
A **miss** is a sample with no unstable mode or with the dominant mode in the electron direction.

In [ ]:
def extent_rows(power):
    if np.isfinite(power.values).all():
        return Extent._compute(power, 0.95)[0].values
    return np.array([float(Extent._compute(row.dropna("theta").expand_dims("sample"), 0.95)[0][0]) if np.isfinite(row.values).any() else np.nan for row in power])


def leaves():
    for case in cases:
        for nbasis in tglf_nbasis:
            for width in code_widths:
                yield "TGLF", case, -1.5, nbasis, 32, width, data_root / "TGLF/Runs/LATIN_HYPERCUBE" / case / "kbm_free" / f"fx_fm1p5_{width_tag(width)}_nb{nbasis}"
        for nbasis, nxgrid in gftm_cells.items():
            for width in code_widths:
                yield "GFTM", case, 0.5, nbasis, nxgrid, width, data_root / "GFTM/Runs/LATIN_HYPERCUBE" / case / "kbm_free" / f"galF_{width_tag(width)}_nb{nbasis}_nx{nxgrid}"


def load_leaf(spec):
    code, case, filter_value, nbasis, nxgrid, width, leaf = spec
    scan = PyroHypercube(pyroscan_json=leaf / "pyroscan.json", load_base_pyro=True)
    scan.load_gk_output(netcdf_file=leaf / "pyroscan.nc")
    flags = json.loads((leaf / "manifest.json").read_text())["code_flags"]
    assert (float(flags["WIDTH"]), float(flags["FILTER"]), int(flags["NBASIS_MAX"]), int(flags["NXGRID"])) == (width, filter_value, nbasis, nxgrid), leaf
    output = scan.gk_output.data
    assert re.sub(r"_pyroscan_base\d+", "", str(output.growth_rate.data.units)) == gs2[case]["units"], "normalisations differ"
    names = [str(name) for name in output.sample_name.values]
    output = output.set_index(sample="sample_name")
    growth = output.growth_rate.pint.dequantify().transpose("sample", "mode")
    assert (growth.fillna(-np.inf).argmax("mode") == 0).all(), leaf     # IBRANCH -1: mode 0 is the dominant mode
    frequency = mc.ION_DIRECTION * output.mode_frequency.pint.dequantify().transpose("sample", "mode")
    row = dict(code=code, case=case, nbasis=nbasis, nxgrid=nxgrid, width=width, sample=names, failed=growth.isnull().all("mode").values,
               gamma=growth.isel(mode=0).fillna(0).values, omega=frequency.isel(mode=0).values)  # mode0-guard-allow: IBRANCH -1, argmax==0 asserted above
    for field in fields:
        power = (np.abs(output[field].isel(mode=0).pint.dequantify().squeeze("kx")) ** 2).transpose("sample", "theta")  # mode0-guard-allow: as above
        row[f"{field}_extent"] = np.where(row["gamma"] > 0, extent_rows(power), np.nan)
    return pd.DataFrame(row)


with get_context("fork").Pool(n_cpu) as pool:
    runs = pd.concat(pool.imap_unordered(load_leaf, list(leaves()), chunksize=4), ignore_index=True)
samples["sample_index"] = np.arange(len(samples))
runs = runs.merge(samples[["case", "sample", "sample_index", "gamma_gs2", "phi_extent_theta", "apar_extent_theta"]], on=["case", "sample"], validate="many_to_one")
print("runs with no output, dropped:", runs.failed.sum(), "of", len(runs), flush=True)
runs = runs[~runs.failed]
runs["miss"] = ~((runs.gamma > 0) & (runs.omega > 0))
width_index = runs.width.map({width: int(np.argmin(np.abs(projection_widths - width))) for width in code_widths}).values
for field in ("phi", "apar", "both"):
    runs[f"projection_{field}"] = error[field][runs.sample_index.values, width_index, runs.nbasis.values - 1]
runs["r_gamma"] = np.where(runs.miss, np.nan, np.log10(runs.gamma / runs.gamma_gs2))
for field in fields:
    runs[f"r_{field}"] = np.where(runs.miss, np.nan, np.log10(runs[f"{field}_extent"] / runs[f"{field}_extent_theta"]))
top_nbasis = {"TGLF": max(tglf_nbasis), "GFTM": max(gftm_cells)}
top = runs[runs.nbasis == runs.code.map(top_nbasis)][["code", "case", "sample", "width", "gamma", "miss"]].rename(columns=dict(gamma="gamma_top", miss="miss_top"))
runs = runs.merge(top, on=["code", "case", "sample", "width"], how="left", validate="many_to_one")
extent_band, gamma_band = np.log10(1.1), 0.05     # "extent agrees": both fields within 10% of GS2; "gamma converged": within 5% of the top NBASIS at the same width
runs["resolved"] = runs.projection_both <= validation_tolerance
runs["resolved_1pct"] = runs.projection_both <= target_tolerance
runs["extent_ok"] = (~runs.miss) & (runs.r_phi.abs() <= extent_band) & (runs.r_apar.abs() <= extent_band)
runs["gamma_converged"] = np.where(runs.miss_top.fillna(True) | (runs.nbasis == runs.code.map(top_nbasis)), np.nan,   # the top rung is not compared with itself
                                    (~runs.miss) & (np.abs(runs.gamma / runs.gamma_top - 1) <= gamma_band))
print(runs.groupby(["code", "nxgrid", "nbasis"]).agg(leaves=("width", "nunique"), rows=("sample", "size")).to_string(), flush=True)

Per code, against WIDTH and one line per NBASIS (Part A format): the fraction of the 738 samples that (left) the projection resolves to 10 % in both
fields (`validation_tolerance`; 1 % is reached by too few samples to draw, its fraction is in `code_width_nbasis_summary.csv`), (middle) the code gives both extents within 10 % of GS2, (right) the code gives $\gamma$ within 5 % of its own top-NBASIS value at that width.

In [ ]:
for code in ("TGLF", "GFTM"):
    data = runs[runs.code == code]
    code_ladder = sorted(data.nbasis.unique())
    code_colours = plt.cm.viridis(np.linspace(0, 0.95, len(code_ladder)))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharex=True, sharey=True, constrained_layout=True)
    for ax, column, title in zip(axes, ("resolved", "extent_ok", "gamma_converged"), ("projection error below 10%", "extents within 10% of GS2", r"$\gamma$ within 5% of top NBASIS")):
        for nbasis, colour in zip(code_ladder, code_colours):
            if column == "gamma_converged" and nbasis == top_nbasis[code]:
                continue
            fraction = data[data.nbasis == nbasis].groupby("width")[column].mean()
            ax.plot(fraction.index, fraction.values, color=colour, marker="o", ms=3, label=f"NBASIS {nbasis}")
        ax.set_xscale("log"); ax.set_title(f"{code}, {title}", fontsize=16); ax.set_xlabel("WIDTH", fontsize=16); ax.tick_params(axis="both", labelsize=14)
        ax.xaxis.set_major_formatter(StrMethodFormatter("{x:g}")); ax.xaxis.set_minor_locator(NullLocator())
    axes[0].set_xticks([0.2, 0.5, 1, 2, 4]); axes[0].set_xlim(0.19, 4.2); axes[0].set_ylim(0, 1)
    axes[0].set_ylabel("fraction of samples", fontsize=16); axes[0].legend(fontsize=12, ncol=2)
    for suffix in ("pdf", "png"):
        fig.savefig(plots_dir / f"validation_fraction_vs_width_{code}.{suffix}")
    plt.close(fig)

# Per-sample predictive power of the projection, and the codes' errors at the rule's cell and at the existing candidates
bins = [0, 0.01, 0.03, 0.1, 0.3, 2]
runs["projection_bin"] = pd.cut(runs.projection_both, bins, include_lowest=True)
conditional = runs.groupby(["code", "projection_bin"], observed=True).agg(cells=("sample", "size"), miss=("miss", "mean"), extent_ok=("extent_ok", "mean"), gamma_converged=("gamma_converged", "mean"),
                                                                           median_abs_r_gamma=("r_gamma", lambda r: r.abs().median()), median_abs_r_phi=("r_phi", lambda r: r.abs().median()),
                                                                           median_abs_r_apar=("r_apar", lambda r: r.abs().median())).reset_index()
conditional.to_csv(csv_dir / "code_outcome_vs_projection_error.csv", index=False)
print(conditional.round(3).to_string())
rms = lambda r: np.sqrt((r.dropna() ** 2).mean())
cells = runs.groupby(["code", "nbasis", "nxgrid", "width"]).agg(n=("sample", "size"), resolved_10pct=("resolved", "mean"), resolved_1pct=("resolved_1pct", "mean"), projection_q50=("projection_both", "median"), projection_q90=("projection_both", lambda e: e.quantile(0.9)),
                                                                 miss=("miss", "mean"), extent_ok=("extent_ok", "mean"), gamma_converged=("gamma_converged", "mean"),
                                                                 gamma_rms=("r_gamma", rms), phi_rms=("r_phi", rms), apar_rms=("r_apar", rms)).reset_index()
cells.to_csv(csv_dir / "code_width_nbasis_summary.csv", index=False)
print(cells[cells.width.isin([0.3031, 0.3482, 0.4, 0.4595, 0.5279, 0.6063, 0.8, 1.0]) & (cells.nbasis >= 8)].round(3).to_string())
samples.drop(columns="sample_index").describe().T.to_csv(csv_dir / "population_summary.csv")